# Knowledge Distillation (KD)

## What is Knowledge Distillation?

**Knowledge Distillation (KD)** transfers the reasoning capacity, class boundaries, or dark knowledge embedded within a massive Teacher model ($T$) into a smaller, computationally lightweight Student model ($S$).

## Classical Logit & Feature Distillation Pipeline

```
               Input Tokens X
                     │
         ┌───────────┴───────────┐
         ▼                       ▼
┌─────────────────┐     ┌─────────────────┐
│  TEACHER (FP16) │     │ STUDENT (INT4)  │
│  (Frozen Params)│     │(Trainable Params)
└────────┬────────┘     └────────┬────────┘
         │                       │
         ├───────────────────────┼───────────────────────┐
         ▼                       ▼                       ▼
  Teacher Logits z_T      Student Logits z_S      Hidden Features h_S
         │                       │                       │
         │  Scale by Temp τ      │  Scale by Temp τ      │ Projection Layer W_p
         ▼                       ▼                       ▼
  Soft Target P_T         Soft Target P_S         Projected Feature h_S'
         │                       │                       │
         └───────────┬───────────┘                       │
                     ▼                                   ▼
          Forward KL Divergence                 Feature Alignment
       D_KL(P_T(x) || P_S(x))                     || h_T - W_p h_S ||²
                     │                                   │
                     └─────────────────┬─────────────────┘
                                       ▼
                             Combined Student Loss L_total
```

## The Core Problem: Large Model → Small Model

Imagine you build a massive AI model—say, a **100 Billion** parameter model.

It's incredibly smart and gets great results, but it's a total resource hog:

* It requires 4 expensive GPUs just to run a single query.
* It takes 2 seconds to generate a response.
* Hosting it costs thousands of dollars a month.

Now you want to train a much smaller, faster model—say, a **7 Billion** parameter model—so it can run cheaply on a single GPU or even locally on a device.

Normally, you would train this 7B model from scratch using raw data (like Wikipedia or books). But a 7B model trained from scratch will never be as smart as the 100B model because it lacks the capacity to discover deep, complex patterns on its own.

### The Distillation Solution: Teacher vs. Student

Instead of letting the small model learn entirely on its own, we pair them up:

* **The Teacher:** The huge, expensive 100B model (smart, but slow).
* **The Student:** The small 7B model (fast, but needs guidance).

**Knowledge Distillation** is the process of using the Teacher model to "tutor" the Student model so the Student reaches near-Teacher performance while staying small and fast.

## How Does the Tutor Actually Teach? (The "Secret Ingredient")

If you give a student a multiple-choice exam, there are two ways to grade it:

### 1. Hard Labels (Standard Training)

You just give the student the answer key:

```
Question 1: A
Question 2: C
```

The student learns what the right answer is, but gets zero insight into why or how close the other options were.

### 2. Soft Labels / "Dark Knowledge" (Distillation)

Instead of just handing over the answer key, the Teacher model reveals its full probability distribution across all choices.

Suppose the Teacher is looking at a picture of a Labrador:

```
Dog: 85% probability
Cat: 14% probability (It's furry, has 4 legs, lives in a house)
Car: 0.00001% probability
```

Notice that **14% on "Cat"**! That tiny percentage contains massive information:
* It tells the student: "Hey, if you aren't sure it's a dog, a cat is a much better guess than a car."

This extra information hidden inside the non-correct answers is what **Geoff Hinton** called **"Dark Knowledge."**

When the Student model is forced to match the Teacher's entire probability curve (instead of just picking the top answer), it learns the Teacher's internal logic much faster.

## The Step-by-Step Distillation Workflow

Imagine we are training our small Student model on a dataset of text prompts or images. For every single input, here is the 4-step pipeline:

### The Distillation Pipeline

```
                       Input Prompt / Image (X)
                                 │
                 ┌───────────────┴───────────────┐
                 ▼                               ▼
       1. Teacher Forward Pass         2. Student Forward Pass
        (Large Model - FROZEN)          (Small Model - TRAINABLE)
                 │                               │
                 ▼                               ▼
       Teacher Raw Scores (Logits)      Student Raw Scores (Logits)
       [ Dog: 12.0, Cat: 5.0, ... ]     [ Dog: 2.1, Cat: 0.8, ... ]
                 │                               │
                 └───────────────┬───────────────┘
                                 ▼
                     3. Apply Temperature (τ)
                        Softens both distributions
                                 │
                                 ▼
                     4. Calculate Distillation Loss
                        Measures gap between Teacher & Student
                                 │
                                 ▼
                     Backpropagate into Student
                     (Teacher stays untouched!)
```

## Steps 1 & 2: Forward Passes

### Step 1: Pass the Input Through the Teacher

We feed an input (e.g., a sentence or image) into the Teacher.

* The Teacher is frozen (its weights are locked; we are not training it).
* The Teacher outputs its raw prediction scores (called **Logits**).

### Step 2: Pass the Same Input Through the Student

We feed the exact same input into the Student.

* The Student is trainable.
* The Student outputs its own raw prediction scores (**Logits**). Because it's still learning, its initial predictions will be rough or wrong.

## Step 3: Apply "Temperature" ($\tau$) to Soften Predictions

This is where the magic happens.

Normally, a neural network uses a standard softmax function to turn raw scores into percentages. But standard softmax makes the top prediction super high (like $99.9\%$) and pushes all other options down to almost $0.0\%$. This hides the "Dark Knowledge"!

To fix this, we divide the scores by a **Temperature** hyperparameter ($\tau$):

* **Temperature = 1 (Normal):** High confidence, sharp peak.
  ```
  [ Dog: 99.9%, Cat: 0.09%, Car: 0.01% ]
  ```

* **Temperature = 3 or 5 (Softened):** Flattens the curve so secondary options become visible.
  ```
  [ Dog: 70.0%, Cat: 25.0%, Car: 5.0% ]
  ```

By turning up the temperature, we force the Teacher to reveal how it compares secondary options. We apply this same temperature softening to the Student.

## Step 4: Calculate the Loss & Update the Student

Now we compare the Teacher's softened predictions with the Student's softened predictions using a mathematical metric (usually **KL-Divergence**).

The loss function asks a simple question:

*"How different is the Student's probability curve from the Teacher's probability curve?"*

* If the Student's curve matches the Teacher's curve $\implies$ Loss is low.
* If the Student's curve is totally different $\implies$ Loss is high.

We take that loss, calculate gradients, and update **ONLY** the Student's weights via backpropagation.

## The Dual-Loss Strategy (The Total Score)

In real-world production, we usually don't rely only on the Teacher. We give the Student two signals at the same time:

$$\text{Total Loss} = \text{Match the Teacher (Soft Loss)} + \text{Get the Right Ground-Truth Answer (Hard Loss)}$$

* **Teacher Matching Loss (Soft Loss):** Guides the Student on how to reason like the Teacher across all choices.
* **Ground-Truth Target Loss (Hard Loss):** Keeps the Student anchored to the actual correct answer key so it doesn't learn the Teacher's mistakes.

## Where in the Model Does Matching Take Place?

When distilling a model, we aren't restricted to looking only at the final output answers. We can align the Teacher and Student across three distinct levels of their internal architecture:

### Three Levels of Distillation Matching

```
                 Teacher Model                     Student Model
             ┌──────────────────┐               ┌──────────────────┐
Level 3 ────►│   Output Layer   │  ◄───────────►│   Output Layer   │ (Logits)
             ├──────────────────┤               ├──────────────────┤
             │   Transformer    │               │   Transformer    │
Level 2 ────►│     Layer 24     │  ◄──[Proj]───►│     Layer 12     │ (Hidden Features)
             │   Transformer    │               │   Transformer    │
Level 1 ────►│   Attention L12  │  ◄───────────►│   Attention L6   │ (Attention Maps)
             └──────────────────┘               └──────────────────┘
```

## The Three Distillation Levels

### Level 3: Output Logit Distillation (The Standard Approach)

**Where:** At the very end of the model (the output logits/probabilities).

**What is matched:** The final probability distribution over classes or tokens.

**How it works:** The Student tries to output the same final answer probabilities as the Teacher.

**Pros:** Simple to implement; model internal architectures don't need to match at all.

**Cons:** Treats the inside of the model as a black box—the Student only sees the final result, not how the Teacher processed the input.

---

### Level 2: Hidden Feature Distillation (Intermediate Layers / "FitNets")

**Where:** Inside the middle layers of the network (e.g., matching Teacher's Layer 24 representation to Student's Layer 12).

**What is matched:** The internal representation vectors (hidden states) that encode concepts and context.

**The Catch:** The Teacher is big (e.g., hidden dimension $D_T = 4096$), but the Student is small (e.g., hidden dimension $D_S = 1024$). You can't directly compare a 4096-sized vector to a 1024-sized vector!

**The Solution:** We attach a small, learnable **Projection Matrix** ($W_p$) to the Student's hidden layer. This matrix multiplies the Student's $1024$ values to stretch them up to $4096$ so they can be compared directly against the Teacher's values using **Mean Squared Error** ($\text{MSE}$).

---

### Level 1: Attention Structure Distillation

**Where:** Inside the Transformer's Self-Attention heads.

**What is matched:** The **Attention Maps** (which tokens the model pays attention to when reading a sentence).

**How it works:** Forces the Student to "look" at the exact same words or visual regions as the Teacher. If the Teacher pays heavy attention to the relationship between the subject and verb in a sentence, the Student is forced to copy that attention pattern.

---

### Summary Comparison of the 3 Levels

| Distillation Level | What Gets Compared | Main Advantage | Challenge |
| --- | --- | --- | --- |
| **Output Logits (Level 3)** | Final token/class probabilities | Easiest to set up; architecture independent | Misses intermediate reasoning steps |
| **Hidden Features (Level 2)** | Internal hidden state vectors | Guides student layer-by-layer | Requires projection layers to fix size differences |
| **Attention Maps (Level 1)** | Query-Key attention matrices | Copies how the model processes context | Harder to pair up heads if count differs |